# Diabetic Prediction Pre-processing

This notebook follows the roadmap for Phase 1 and Phase 2 of the diabetic prediction project.

Phase 1 covers the problem framing and environment setup.
Phase 2 covers data loading, inspection, and cleaning.

## Phase 1 - Problem Setup

- Define the task as a binary classification problem.
- Prepare the notebook environment and project paths.
- Identify the source dataset before any cleaning starts.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent

raw_data_dir = project_root / 'data' / 'raw'
processed_data_dir = project_root / 'data' / 'processed'
raw_data_dir.mkdir(parents=True, exist_ok=True)
processed_data_dir.mkdir(parents=True, exist_ok=True)

candidate_files = sorted(raw_data_dir.glob('*.csv'))
raw_data_path = candidate_files[0] if candidate_files else None

print(f'Project root: {project_root}')
print(f'Raw data folder: {raw_data_dir}')
print(f'Processed data folder: {processed_data_dir}')
print(f'Raw dataset: {raw_data_path.name if raw_data_path else "No CSV found in data/raw"}')

Project root: F:\GIT_Projects\diabetic prediction\diabetic-prediction
Raw data folder: F:\GIT_Projects\diabetic prediction\diabetic-prediction\data\raw
Processed data folder: F:\GIT_Projects\diabetic prediction\diabetic-prediction\data\processed
Raw dataset: diabetes.csv


## Phase 2 - Data Preparation

- Load the diabetes dataset from `data/raw/`.
- Inspect its shape, columns, and missing values.
- Clean the data and save a processed copy for modeling.

In [2]:
if raw_data_path is None:
    raise FileNotFoundError(
        'No CSV file was found in data/raw. Add the diabetes dataset there before running Phase 2.'
    )

df = pd.read_csv(raw_data_path)

print(f'Dataset shape: {df.shape}')
display(df.head())
display(df.info())
display(df.isna().sum().to_frame('missing_values').T)

Dataset shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


None

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
missing_values,0,0,0,0,0,0,0,0,0


In [3]:
cleaned_df = df.copy()

zero_as_missing_columns = [
    column for column in ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
    if column in cleaned_df.columns
]

for column in zero_as_missing_columns:
    cleaned_df[column] = cleaned_df[column].replace(0, np.nan)

for column in zero_as_missing_columns:
    if cleaned_df[column].isna().any():
        cleaned_df[column] = cleaned_df[column].fillna(cleaned_df[column].median())

if 'Outcome' in cleaned_df.columns:
    cleaned_df['Outcome'] = cleaned_df['Outcome'].astype(int)

processed_data_path = processed_data_dir / 'diabetes_clean.csv'
cleaned_df.to_csv(processed_data_path, index=False)

print(f'Processed dataset saved to: {processed_data_path}')
print(f'Cleaned shape: {cleaned_df.shape}')
display(cleaned_df.head())
display(cleaned_df.isna().sum().to_frame('missing_values').T)

Processed dataset saved to: F:\GIT_Projects\diabetic prediction\diabetic-prediction\data\processed\diabetes_clean.csv
Cleaned shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.0,72.0,35.0,125.0,33.6,0.627,50,1
1,1,85.0,66.0,29.0,125.0,26.6,0.351,31,0
2,8,183.0,64.0,29.0,125.0,23.3,0.672,32,1
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21,0
4,0,137.0,40.0,35.0,168.0,43.1,2.288,33,1


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
missing_values,0,0,0,0,0,0,0,0,0
